# All-in-One Report Experiments

Notebook nay chay mot bo experiment khoang 8-10 gio tren Kaggle T4x2 va xuat bang ket qua dung truc tiep trong bao cao.

Bo mac dinh gom cac ablation co kiem soat: prompt basic vs strict, LoRA r8 vs r16, context 768 vs 1024, all-on medium, va DeepSeek comparison.


In [1]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time
from zipfile import ZipFile

PROJECT_NAME = 'pretrained-summarization'
REPO_URL = 'https://github.com/Anhnguyen0812/pretrained-summarization.git'
REFRESH_REPO = True
WORKING = Path('/kaggle/working')
WORKING_REPO = WORKING / PROJECT_NAME
OUTPUT_ROOT = WORKING / 'report_experiment_outputs'
REPORT_DIR = OUTPUT_ROOT / '_report'
DATA_DIR = Path('/kaggle/input/datasets/anhnguyen0812/nlp-vietnamese-sumarization')
TRAIN_FILE = DATA_DIR / 'train-00000-of-00001.parquet'
VALID_FILE = DATA_DIR / 'valid-00000-of-00001.parquet'

os.chdir(WORKING)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, check=True):
    print('CMD:', cmd, flush=True)
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    process = subprocess.Popen(cmd, shell=True, cwd=str(cwd) if cwd else None, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in process.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    code = process.wait()
    if check and code != 0:
        raise RuntimeError(f'Command failed with exit code {code}: {cmd}\nLast lines:\n{"".join(lines[-100:])}')
    return code

def is_repo(path):
    return (path / 'pyproject.toml').exists() and (path / 'src' / 'vn_summarization').exists()

if REFRESH_REPO and WORKING_REPO.exists():
    shutil.rmtree(WORKING_REPO)
if not is_repo(WORKING_REPO):
    run(f'git clone --depth 1 {REPO_URL} {WORKING_REPO}', cwd=WORKING)
else:
    run('git pull --ff-only', cwd=WORKING_REPO, check=False)
repo = WORKING_REPO
run('git log --oneline -1', cwd=repo)


CMD: git clone --depth 1 https://github.com/Anhnguyen0812/pretrained-summarization.git /kaggle/working/pretrained-summarization
Cloning into '/kaggle/working/pretrained-summarization'...
CMD: git log --oneline -1
f3e3433 Revise report notebook for controlled ablations


0

In [2]:
os.chdir(repo)
run(f'{sys.executable} -m pip install -q --upgrade pip', cwd=repo)
run(f'{sys.executable} -m pip install -q -e .', cwd=repo)
run(f'{sys.executable} -m pip install -q --upgrade "transformers>=4.51.0,<5" "tokenizers>=0.22.0,<=0.23.0"', cwd=repo)
run(f'{sys.executable} -m pip check', cwd=repo, check=False)
run(f'{sys.executable} -m pip show transformers tokenizers peft accelerate | sed -n "/Name: /p;/Version: /p"', cwd=repo, check=False)
run(f"{sys.executable} -c 'import tokenizers, transformers; print(\"TRANSFORMERS\", transformers.__version__); print(\"TOKENIZERS\", tokenizers.__version__)'", cwd=repo)


CMD: /usr/bin/python3 -m pip install -q --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 30.5 MB/s eta 0:00:00
CMD: /usr/bin/python3 -m pip install -q -e .
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompa

0

In [3]:
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
run('nvidia-smi', check=False)
print('TRAIN_FILE:', TRAIN_FILE, TRAIN_FILE.exists())
print('VALID_FILE:', VALID_FILE, VALID_FILE.exists())
if not TRAIN_FILE.exists() or not VALID_FILE.exists():
    raise FileNotFoundError('Attach Kaggle dataset first.')

import torch
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
print('NUM_GPUS', NUM_GPUS)


CMD: nvidia-smi
Wed Jun 10 06:10:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------------------------

## Experiment Plan

Cac run nay giu Qwen3 lam model chinh va bat/tat tung ky thuat mot cach co kiem soat. DeepSeek chi la comparison model family.


In [4]:
import yaml

BASE_OVERRIDES = [
    f'data.train_file={TRAIN_FILE}',
    f'data.valid_file={VALID_FILE}',
]
OVERWRITE_RUNS = False
OVERWRITE_EVALS = False

GENERATED_CONFIG_DIR = repo / 'configs' / '_generated_report'
GENERATED_CONFIG_DIR.mkdir(parents=True, exist_ok=True)

def deep_update_dict(base, updates):
    result = dict(base)
    for key, value in updates.items():
        if isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = deep_update_dict(result[key], value)
        else:
            result[key] = value
    return result

def make_config(src, dst_name, updates):
    with (repo / src).open('r', encoding='utf-8') as f:
        data = yaml.safe_load(f)
    data = deep_update_dict(data, updates)
    dst = GENERATED_CONFIG_DIR / dst_name
    with dst.open('w', encoding='utf-8') as f:
        yaml.safe_dump(data, f, allow_unicode=True, sort_keys=False)
    return dst.relative_to(repo).as_posix()

BASIC_PROMPT = 'Tóm tắt văn bản sau bằng tiếng Việt.\n\nVăn bản:\n{article}\n\nTóm tắt:\n'
STRICT_PROMPT = 'Bạn là trợ lý tóm tắt tiếng Việt. Viết duy nhất một đoạn tóm tắt cuối cùng, không viết suy luận, không giải thích, không dùng thẻ <think>. Giữ tên riêng, số liệu, mốc thời gian quan trọng. Không thêm thông tin ngoài văn bản.\n\nVăn bản:\n{article}\n\nTóm tắt:\n'
QWEN3_BASIC_CONFIG = make_config('configs/qwen3_1_7b_lora.yaml', 'qwen3_basic_prompt.yaml', {'data': {'prompt_template': BASIC_PROMPT}})
QWEN3_STRICT_CONFIG = make_config('configs/qwen3_1_7b_lora.yaml', 'qwen3_strict_prompt.yaml', {'data': {'prompt_template': STRICT_PROMPT}})

QWEN3_BASE_R8_768 = [
    'training.num_train_epochs=1',
    'training.max_steps=300',
    'training.eval_steps=300',
    'training.save_steps=300',
    'training.logging_steps=25',
    'training.save_total_limit=1',
    'training.ddp_find_unused_parameters=false',
    'training.per_device_train_batch_size=1',
    'training.per_device_eval_batch_size=2',
    'training.gradient_accumulation_steps=8',
    'data.max_train_samples=2500',
    'data.max_eval_samples=500',
    'data.max_source_length=768',
    'data.max_target_length=128',
    'data.max_length=896',
    'generation.max_new_tokens=128',
    'generation.num_beams=1',
    'lora.r=8',
    'lora.lora_alpha=16',
]

QWEN3_R16_768 = QWEN3_BASE_R8_768[:-2] + [
    'lora.r=16',
    'lora.lora_alpha=32',
]

QWEN3_R8_1024 = [
    item for item in QWEN3_BASE_R8_768
    if not item.startswith(('data.max_source_length=', 'data.max_target_length=', 'data.max_length='))
] + [
    'data.max_source_length=1024',
    'data.max_target_length=128',
    'data.max_length=1152',
]

QWEN3_ALL_ON_MEDIUM = [
    'training.num_train_epochs=1',
    'training.max_steps=700',
    'training.eval_steps=200',
    'training.save_steps=200',
    'training.logging_steps=25',
    'training.save_total_limit=2',
    'training.ddp_find_unused_parameters=false',
    'training.per_device_train_batch_size=1',
    'training.per_device_eval_batch_size=2',
    'training.gradient_accumulation_steps=8',
    'data.max_train_samples=7000',
    'data.max_eval_samples=500',
    'data.max_source_length=1024',
    'data.max_target_length=160',
    'data.max_length=1184',
    'generation.max_new_tokens=160',
    'generation.num_beams=2',
    'lora.r=16',
    'lora.lora_alpha=32',
]

EXPERIMENTS = [
    {
        'name': 'qwen3_prompt_basic_r8_300s_report',
        'kind': 'causal',
        'model_label': 'Qwen3-1.7B',
        'technique': 'Prompt basic + LoRA r8',
        'ablation_focus': 'prompt_engineering_off',
        'config': QWEN3_BASIC_CONFIG,
        'overrides': QWEN3_BASE_R8_768,
        'expected_time': '1-1.5h',
    },
    {
        'name': 'qwen3_prompt_strict_r8_300s_report',
        'kind': 'causal',
        'model_label': 'Qwen3-1.7B',
        'technique': 'Prompt strict + LoRA r8',
        'ablation_focus': 'prompt_engineering_on',
        'config': QWEN3_STRICT_CONFIG,
        'overrides': QWEN3_BASE_R8_768,
        'expected_time': '1-1.5h',
    },
    {
        'name': 'qwen3_lora_r16_300s_report',
        'kind': 'causal',
        'model_label': 'Qwen3-1.7B',
        'technique': 'Prompt strict + LoRA r16',
        'ablation_focus': 'lora_rank_on_r16_vs_r8',
        'config': QWEN3_STRICT_CONFIG,
        'overrides': QWEN3_R16_768,
        'expected_time': '1.5-2h',
    },
    {
        'name': 'qwen3_context1024_r8_300s_report',
        'kind': 'causal',
        'model_label': 'Qwen3-1.7B',
        'technique': 'Prompt strict + longer context',
        'ablation_focus': 'long_context_on_1024_vs_768',
        'config': QWEN3_STRICT_CONFIG,
        'overrides': QWEN3_R8_1024,
        'expected_time': '1.5-2h',
    },
    {
        'name': 'qwen3_all_on_r16_700s_report',
        'kind': 'causal',
        'model_label': 'Qwen3-1.7B',
        'technique': 'Prompt strict + r16 + context1024 + more data',
        'ablation_focus': 'combined_best_setting',
        'config': QWEN3_STRICT_CONFIG,
        'overrides': QWEN3_ALL_ON_MEDIUM,
        'expected_time': '3-4h',
    },
    {
        'name': 'deepseek_lora_r8_300s_report',
        'kind': 'causal',
        'model_label': 'DeepSeek-R1-Distill-Qwen-1.5B',
        'technique': 'Prompt strict + LoRA r8',
        'ablation_focus': 'model_family_vs_qwen3_fast',
        'config': 'configs/deepseek_r1_distill_qwen_1_5b_lora.yaml',
        'overrides': QWEN3_BASE_R8_768,
        'expected_time': '1-1.5h',
    },
]
print(json.dumps(EXPERIMENTS, indent=2, ensure_ascii=False))


[
  {
    "name": "qwen3_prompt_basic_r8_300s_report",
    "kind": "causal",
    "model_label": "Qwen3-1.7B",
    "technique": "Prompt basic + LoRA r8",
    "ablation_focus": "prompt_engineering_off",
    "config": "configs/_generated_report/qwen3_basic_prompt.yaml",
    "overrides": [
      "training.num_train_epochs=1",
      "training.max_steps=300",
      "training.eval_steps=300",
      "training.save_steps=300",
      "training.logging_steps=25",
      "training.save_total_limit=1",
      "training.ddp_find_unused_parameters=false",
      "training.per_device_train_batch_size=1",
      "training.per_device_eval_batch_size=2",
      "training.gradient_accumulation_steps=8",
      "data.max_train_samples=2500",
      "data.max_eval_samples=500",
      "data.max_source_length=768",
      "data.max_target_length=128",
      "data.max_length=896",
      "generation.max_new_tokens=128",
      "generation.num_beams=1",
      "lora.r=8",
      "lora.lora_alpha=16"
    ],
    "expected_ti

In [5]:
def _override_string(items):
    return ' '.join(f'--set {item}' for item in items)

def latest_checkpoint(run_dir):
    checkpoints = sorted(run_dir.glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[-1]) if p.name.split('-')[-1].isdigit() else -1)
    return checkpoints[-1] if checkpoints else None

def launch_train(module, module_args):
    train_cmd = f'-m {module} {module_args}'
    if NUM_GPUS >= 2:
        cmd = f'{sys.executable} -m accelerate.commands.launch --multi_gpu --num_processes {NUM_GPUS} --num_machines 1 --mixed_precision fp16 --dynamo_backend no {train_cmd}'
    else:
        cmd = f'{sys.executable} -u {train_cmd}'
    return run(cmd, cwd=repo)

def run_python_module(module, module_args):
    return run(f'{sys.executable} -u -m {module} {module_args}', cwd=repo)

def prepare_train_dir(run_dir, overwrite=False):
    resume = []
    if run_dir.exists() and not overwrite:
        ckpt = latest_checkpoint(run_dir)
        if ckpt:
            print('RESUME', ckpt)
            resume = [f'training.resume_from_checkpoint={ckpt}']
        elif not (run_dir / 'best').exists():
            print('CLEAR incomplete run:', run_dir)
            shutil.rmtree(run_dir)
    return resume

def train_experiment(exp):
    run_dir = OUTPUT_ROOT / exp['name']
    if (run_dir / 'best' / 'adapter_config.json').exists() and not OVERWRITE_RUNS:
        print('SKIP train adapter exists:', run_dir / 'best')
        return 0
    if (run_dir / 'best' / 'config.json').exists() and not OVERWRITE_RUNS:
        print('SKIP train model exists:', run_dir / 'best')
        return 0
    resume = prepare_train_dir(run_dir, OVERWRITE_RUNS)
    final_overrides = BASE_OVERRIDES + [f'training.output_dir={run_dir}'] + resume + exp['overrides']
    module = 'vn_summarization.train_causal_lm' if exp['kind'] == 'causal' else 'vn_summarization.train'
    args = f'--config {exp["config"]} {_override_string(final_overrides)}'
    return launch_train(module, args)

def eval_experiment(exp):
    eval_dir = OUTPUT_ROOT / (exp['name'] + '_eval')
    if (eval_dir / 'validation_metrics.json').exists() and not OVERWRITE_EVALS:
        print('SKIP eval:', eval_dir / 'validation_metrics.json')
        return 0
    eval_dir.mkdir(parents=True, exist_ok=True)
    model_path = OUTPUT_ROOT / exp['name'] / 'best'
    final_overrides = BASE_OVERRIDES + [f'training.output_dir={eval_dir}'] + exp['overrides']
    predictions = eval_dir / 'predictions_valid.jsonl'
    if exp['kind'] == 'causal':
        module = 'vn_summarization.evaluate_causal_lm'
    else:
        module = 'vn_summarization.evaluate'
    args = f'--config {exp["config"]} --model_path {model_path} --predictions_path {predictions} {_override_string(final_overrides)}'
    return run_python_module(module, args)


## Run Train/Eval

Moi experiment train xong se evaluate generation tren validation subset roi ghi predictions. Neu notebook bi ngat, chay lai se resume/skip run da xong.


In [6]:
start_all = time.time()
for exp in EXPERIMENTS:
    print('\n' + '=' * 100)
    print('EXPERIMENT', exp['name'], exp['technique'], exp['expected_time'])
    t0 = time.time()
    train_experiment(exp)
    eval_experiment(exp)
    print('DONE', exp['name'], 'elapsed_min=', round((time.time() - t0) / 60, 2))
print('ALL elapsed_hours=', round((time.time() - start_all) / 3600, 2))



EXPERIMENT qwen3_prompt_basic_r8_300s_report Prompt basic + LoRA r8 1-1.5h
CMD: /usr/bin/python3 -m accelerate.commands.launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision fp16 --dynamo_backend no -m vn_summarization.train_causal_lm --config configs/_generated_report/qwen3_basic_prompt.yaml --set data.train_file=/kaggle/input/datasets/anhnguyen0812/nlp-vietnamese-sumarization/train-00000-of-00001.parquet --set data.valid_file=/kaggle/input/datasets/anhnguyen0812/nlp-vietnamese-sumarization/valid-00000-of-00001.parquet --set training.output_dir=/kaggle/working/report_experiment_outputs/qwen3_prompt_basic_r8_300s_report --set training.num_train_epochs=1 --set training.max_steps=300 --set training.eval_steps=300 --set training.save_steps=300 --set training.logging_steps=25 --set training.save_total_limit=1 --set training.ddp_find_unused_parameters=false --set training.per_device_train_batch_size=1 --set training.per_device_eval_batch_size=2 --set training.gradient_acc

In [7]:
def load_json(path):
    if not path.exists():
        return {}
    with path.open('r', encoding='utf-8') as f:
        return json.load(f)

def pick_metric(metrics, name):
    for key in [name, f'eval_{name}', f'test_{name}', f'predict_{name}']:
        if key in metrics:
            return metrics[key]
    for key, value in metrics.items():
        if key.endswith('_' + name):
            return value
    return ''

def override_value(overrides, key):
    prefix = key + '='
    for item in overrides:
        if item.startswith(prefix):
            return item.split('=', 1)[1]
    return ''

def as_float(value):
    try:
        return float(value)
    except Exception:
        return float('-inf')

rows = []
for exp in EXPERIMENTS:
    run_dir = OUTPUT_ROOT / exp['name']
    eval_dir = OUTPUT_ROOT / (exp['name'] + '_eval')
    metrics_path = eval_dir / 'validation_metrics.json'
    train_metrics_path = run_dir / 'train_results.json'
    metrics = load_json(metrics_path)
    train_metrics = load_json(train_metrics_path)
    row = {
        'run': exp['name'],
        'model': exp['model_label'],
        'kind': exp['kind'],
        'technique': exp['technique'],
        'ablation_focus': exp['ablation_focus'],
        'expected_time': exp['expected_time'],
        'max_steps': override_value(exp['overrides'], 'training.max_steps'),
        'train_samples': override_value(exp['overrides'], 'data.max_train_samples'),
        'eval_samples': override_value(exp['overrides'], 'data.max_eval_samples'),
        'source_len': override_value(exp['overrides'], 'data.max_source_length'),
        'target_len': override_value(exp['overrides'], 'data.max_target_length'),
        'lora_r': override_value(exp['overrides'], 'lora.r'),
        'beams': override_value(exp['overrides'], 'generation.num_beams'),
        'rouge1': pick_metric(metrics, 'rouge1'),
        'rouge2': pick_metric(metrics, 'rouge2'),
        'rougeL': pick_metric(metrics, 'rougeL'),
        'loss': pick_metric(metrics, 'loss'),
        'gen_len': pick_metric(metrics, 'gen_len'),
        'train_runtime': pick_metric(train_metrics, 'runtime'),
        'metrics_file': metrics_path.relative_to(WORKING).as_posix() if metrics_path.exists() else '',
        'predictions_file': (eval_dir / 'predictions_valid.jsonl').relative_to(WORKING).as_posix() if (eval_dir / 'predictions_valid.jsonl').exists() else '',
    }
    rows.append(row)

rows_sorted = sorted(rows, key=lambda row: as_float(row.get('rougeL')), reverse=True)
columns = ['run', 'model', 'kind', 'technique', 'ablation_focus', 'max_steps', 'train_samples', 'eval_samples', 'source_len', 'target_len', 'lora_r', 'beams', 'rouge1', 'rouge2', 'rougeL', 'loss', 'gen_len', 'train_runtime', 'metrics_file', 'predictions_file']

import csv
csv_path = REPORT_DIR / 'report_table.csv'
with csv_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()
    for row in rows_sorted:
        writer.writerow({col: row.get(col, '') for col in columns})

def fmt(value):
    if isinstance(value, float):
        return f'{value:.4f}'
    return str(value)

md_columns = ['run', 'model', 'technique', 'ablation_focus', 'max_steps', 'train_samples', 'eval_samples', 'lora_r', 'beams', 'rouge1', 'rouge2', 'rougeL', 'loss', 'gen_len']
lines = ['# Report Experiment Table', '', '| ' + ' | '.join(md_columns) + ' |', '| ' + ' | '.join(['---'] * len(md_columns)) + ' |']
for row in rows_sorted:
    lines.append('| ' + ' | '.join(fmt(row.get(col, '')) for col in md_columns) + ' |')
lines.append('')
lines.append('Best run by rougeL: ' + (rows_sorted[0]['run'] if rows_sorted else ''))
md_path = REPORT_DIR / 'report_table.md'
md_path.write_text('\n'.join(lines), encoding='utf-8')
best_path = REPORT_DIR / 'best_run.json'
best_path.write_text(json.dumps(rows_sorted[0] if rows_sorted else {}, ensure_ascii=False, indent=2), encoding='utf-8')
print(md_path.read_text(encoding='utf-8'))
print('CSV:', csv_path)
print('Best:', best_path)


# Report Experiment Table

| run | model | technique | ablation_focus | max_steps | train_samples | eval_samples | lora_r | beams | rouge1 | rouge2 | rougeL | loss | gen_len |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| qwen3_context1024_r8_300s_report | Qwen3-1.7B | Prompt strict + longer context | long_context_on_1024_vs_768 | 300 | 2500 | 500 | 8 | 1 | 66.5238 | 31.2174 | 37.9235 |  | 81.3340 |
| qwen3_all_on_r16_700s_report | Qwen3-1.7B | Prompt strict + r16 + context1024 + more data | combined_best_setting | 700 | 7000 | 500 | 16 | 2 | 68.3372 | 32.1273 | 37.8748 |  | 89.3220 |
| qwen3_lora_r16_300s_report | Qwen3-1.7B | Prompt strict + LoRA r16 | lora_rank_on_r16_vs_r8 | 300 | 2500 | 500 | 16 | 1 | 65.6495 | 29.8698 | 36.1109 |  | 83.7900 |
| qwen3_prompt_basic_r8_300s_report | Qwen3-1.7B | Prompt basic + LoRA r8 | prompt_engineering_off | 300 | 2500 | 500 | 8 | 1 | 59.3828 | 27.2874 | 33.7926 |  | 71.9320 |
| qwen3_prompt_strict_r8_300

In [8]:
zip_path = WORKING / 'allinone_report_results.zip'
if zip_path.exists():
    zip_path.unlink()
keep_suffixes = {'.json', '.jsonl', '.csv', '.md', '.txt'}
files = [p for p in OUTPUT_ROOT.rglob('*') if p.is_file() and p.suffix in keep_suffixes]
with ZipFile(zip_path, 'w') as zf:
    for file in files:
        zf.write(file, file.relative_to(WORKING).as_posix())
print('ZIP', zip_path)
for file in sorted(files):
    print(file)


ZIP /kaggle/working/allinone_report_results.zip
/kaggle/working/report_experiment_outputs/_report/best_run.json
/kaggle/working/report_experiment_outputs/_report/report_table.csv
/kaggle/working/report_experiment_outputs/_report/report_table.md
/kaggle/working/report_experiment_outputs/deepseek_lora_r8_300s_report/all_results.json
/kaggle/working/report_experiment_outputs/deepseek_lora_r8_300s_report/best/README.md
/kaggle/working/report_experiment_outputs/deepseek_lora_r8_300s_report/best/adapter_config.json
/kaggle/working/report_experiment_outputs/deepseek_lora_r8_300s_report/best/special_tokens_map.json
/kaggle/working/report_experiment_outputs/deepseek_lora_r8_300s_report/best/tokenizer.json
/kaggle/working/report_experiment_outputs/deepseek_lora_r8_300s_report/best/tokenizer_config.json
/kaggle/working/report_experiment_outputs/deepseek_lora_r8_300s_report/checkpoint-300/README.md
/kaggle/working/report_experiment_outputs/deepseek_lora_r8_300s_report/checkpoint-300/adapter_config